# 05 — Evaluation: Reproducible Paper-Quality Runs

노트북은 실행 순서만 정의하고, 핵심 로직은 `src/evaluation.py`, `src/phase.py`, `src/experiment_plots.py`에 둔다.

**평가 항목**
- Table 1: Vanilla / Periodic / Trajectory in-distribution 성능
- Table 2: Periodic / Trajectory frequency controllability sweep (5 in-dist + 4 OOD)
- Table 3: Periodic / Trajectory phase-offset sweep (`phase0 ∈ {0, π/2, π, 3π/2}`)
- Figure 1: Survival / reward vs frequency
- Raw arrays: `eval_results.npz`


## 1. Setup

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    pass

import sys
from pathlib import Path

REPO_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next((p for p in REPO_ROOT_CANDIDATES if (p / 'src' / 'paths.py').exists()), None)
assert REPO_ROOT is not None, 'repo root with src/paths.py not found; run this notebook from the cloned repository'
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from paths import ARTIFACT_ROOT, DATA_DIR, CHECKPOINTS_DIR, RESULTS_DIR, FIGURES_DIR, ensure_artifact_dirs
from reproducibility import describe_project_paths

ensure_artifact_dirs()
describe_project_paths(REPO_ROOT, SRC_DIR, ARTIFACT_ROOT)


In [ ]:
!pip install -q -r {REPO_ROOT / 'requirements.txt'}
print('✓ requirements.txt 기반 의존성 준비 완료')


In [ ]:
import gymnasium as gym
import torch

from dataset import load_project_data
from evaluation import (
    build_eval_results_payload,
    build_frequency_sweep_protocol,
    build_phase_sweep_protocol,
    load_evaluation_state,
    print_frequency_sweep_summary,
    print_phase_sweep_summary,
    print_table1_summary,
    run_frequency_sweep_evaluation,
    run_in_distribution_evaluation,
    run_phase_sweep_evaluation,
    save_eval_results_npz,
)
from experiment_plots import plot_evaluation_frequency_comparison
from reproducibility import set_global_seed, resolve_device, print_data_summary

device = resolve_device()
print(f'PyTorch {torch.__version__}, device={device}')


## 2. 데이터 및 체크포인트 로드

In [ ]:
data = load_project_data(DATA_DIR)
print_data_summary(data)
set_global_seed(data['seed'], deterministic=True)


In [ ]:
state = load_evaluation_state(data, device=device, checkpoints_dir=CHECKPOINTS_DIR)


## 3. 평가 프로토콜 정의

In [ ]:
DT = 0.05
MAX_STEPS = 1000
N_SEEDS_INDIST = 20
N_SEEDS_SWEEP = 10
N_SEEDS_PHASE = 10

freq_protocol = build_frequency_sweep_protocol(data, n_in_dist=5)
phase_protocol = build_phase_sweep_protocol(data)

print(f"In-dist freqs: {freq_protocol.in_freqs.round(3).tolist()}")
print(f"OOD freqs:     {freq_protocol.ood_freqs.round(3).tolist()}")
print(f"Sweep freqs:   {freq_protocol.sweep_freqs.round(3).tolist()}")
print(f"Zones:         {freq_protocol.zone_labels.tolist()}")
print(f"Phase offsets: {phase_protocol.phase_labels.tolist()} @ f={phase_protocol.freq_hz:.3f} Hz")


## 4. Table 1 — In-distribution performance

In [ ]:
env = gym.make('Ant-v5')
table1_results = run_in_distribution_evaluation(
    state,
    env=env,
    data=data,
    device=device,
    n_seeds=N_SEEDS_INDIST,
    max_steps=MAX_STEPS,
    dt=DT,
)


In [ ]:
print_table1_summary(state, table1_results, freq_hz=float(data['freq_window_mean']))


## 5. Table 2 — Frequency controllability sweep

In [ ]:
freq_results = run_frequency_sweep_evaluation(
    state,
    freq_protocol,
    env=env,
    data=data,
    device=device,
    n_seeds=N_SEEDS_SWEEP,
    max_steps=MAX_STEPS,
    dt=DT,
)


In [ ]:
print_frequency_sweep_summary(data, freq_protocol, freq_results)


## 6. Table 3 — Phase-offset sweep

In [ ]:
phase_results = run_phase_sweep_evaluation(
    state,
    phase_protocol,
    env=env,
    data=data,
    device=device,
    n_seeds=N_SEEDS_PHASE,
    max_steps=MAX_STEPS,
    dt=DT,
)


In [ ]:
print_phase_sweep_summary(phase_protocol, phase_results)


## 7. Figure 1 — Reward vs frequency

In [ ]:
plot_evaluation_frequency_comparison(
    table1_results,
    freq_results,
    data,
    FIGURES_DIR / 'eval_figure1_reward_vs_freq.png',
    n_seeds_sweep=N_SEEDS_SWEEP,
)


## 8. 결과 저장 — `eval_results.npz`

In [ ]:
eval_payload = build_eval_results_payload(
    data,
    table1_results,
    freq_protocol,
    freq_results,
    phase_protocol,
    phase_results,
    n_seeds_indist=N_SEEDS_INDIST,
    n_seeds_sweep=N_SEEDS_SWEEP,
    n_seeds_phase=N_SEEDS_PHASE,
)
save_eval_results_npz(eval_payload, RESULTS_DIR / 'eval_results.npz')


## 9. 완료 체크

- 세 모델 동일 protocol로 측정
- Frequency controllability와 phase-offset sensitivity를 모두 저장
- Notebook은 orchestration만 담당하고 재사용 로직은 `src`에 위치
